In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'yt-dlp>=2024.11.18',
    'huggingface-hub>=0.26.0',
    'python-dotenv>=1.0.0',
    'pyyaml>=6.0',
    'requests>=2.32.0',
    'soundfile>=0.12.1',
    'numpy>=1.26.0',
], check=True)
subprocess.run(['apt-get', 'install', '-qq', '-y', 'ffmpeg'], check=True)

In [ ]:
import os
import json
import time
import shutil
import threading
import subprocess
import sys
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

import yaml
import requests
import soundfile as sf
import numpy as np
from huggingface_hub import HfApi

# --- Custom exception for circuit-breaker (Fix 3) ---
class CookieExpiredError(Exception):
    """Raised when consecutive cookie-rejected downloads indicate cookies are fully expired."""
    pass

WORK_DIR       = Path('/kaggle/working')
DOWNLOAD_DIR   = WORK_DIR / 'raw_downloads'
STANDARD_DIR   = WORK_DIR / 'standardized'
CHECKPOINT_PATH = WORK_DIR / 'checkpoint_p1b.json'
FAILED_LOG     = WORK_DIR / 'failed_downloads.txt'

# --- FIX #1: Multiple cookie source candidates with fallback ---
CONFIG_DIR     = Path('/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2/config')
COOKIES_PATH   = WORK_DIR / 'cookies.txt'
COOKIE_CANDIDATES = [
    CONFIG_DIR / 'cookies.txt',
    Path('/kaggle/working/config/cookies.txt'),
    WORK_DIR / 'cookies.txt',
]

COOKIES_SRC = None
for candidate in COOKIE_CANDIDATES:
    if candidate.exists():
        COOKIES_SRC = candidate
        break

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
STANDARD_DIR.mkdir(parents=True, exist_ok=True)


# --- FIX #2: Always overwrite cookies (avoid stale cookies from previous runs) ---
if COOKIES_SRC is not None:
    shutil.copy2(str(COOKIES_SRC), str(COOKIES_PATH))
    print(f'[config] cookies refreshed from {COOKIES_SRC} -> {COOKIES_PATH}')
else:
    print('[config] WARNING: cookies file NOT FOUND in any location -- YouTube 403s are expected!')
    print('[config] Searched: ' + ', '.join(str(p) for p in COOKIE_CANDIDATES))

TARGET_SR      = 24000
DOWNLOAD_WORKERS = 2
MAX_RETRIES    = 5
RATE_LIMIT_DELAY = 5
SAVE_EVERY     = 20
MIN_DUR_SEC    = 100
MAX_DUR_SEC    = 10800
MIN_FILE_BYTES = 50_000

# --- Fix 2: Mid-session cookie refresh every N downloads ---
COOKIE_REFRESH_EVERY = 100
_download_counter = 0
_cookie_lock = threading.Lock()

# --- Shared thread-safe consecutive cookie-rejection counter ---
# Workers check this before wasting time on retries when cookies are clearly dead
_consecutive_cookie_rejections = 0
_circuit_breaker_tripped = False
COOKIE_CIRCUIT_BREAKER = 5   # trip after 5 consecutive cookie rejections (was 20, too slow)


In [ ]:
def load_secrets():
    try:
        from kaggle_secrets import UserSecretsClient
        c = UserSecretsClient()
        secrets = {
            'HF_TOKEN_PRIMARY':   c.get_secret('HF_TOKEN_PRIMARY'),
            'HF_TOKEN_SECONDARY': c.get_secret('HF_TOKEN_SECONDARY'),
            'HF_TOKEN_TERTIARY':  c.get_secret('HF_TOKEN_TERTIARY'),
            'GEMINI_API_KEY':  c.get_secret('GEMINI_API_KEY_01') or c.get_secret('GEMINI_API_KEY'),
            'PROXY_URL':          c.get_secret('PROXY_URL'),
        }
        print('[secrets] loaded from Kaggle Secrets')
        return secrets
    except Exception:
        pass

    env_file = Path('.env')
    if env_file.exists():
        from dotenv import load_dotenv
        load_dotenv(env_file)
        print('[secrets] loaded from .env')

    required = ['HF_TOKEN_PRIMARY', 'HF_TOKEN_SECONDARY', 'HF_TOKEN_TERTIARY']
    missing = [k for k in required if not os.environ.get(k)]
    if missing:
        raise RuntimeError(f'Missing secrets: {missing}')
    secrets = {k: os.environ[k] for k in required}
    secrets['PROXY_URL'] = os.environ.get('PROXY_URL', '')
    return secrets

SECRETS   = load_secrets()
HF_TOKEN  = SECRETS['HF_TOKEN_PRIMARY']
PROXY_URL = (SECRETS.get('PROXY_URL') or '').strip()
if PROXY_URL:
    print(f'[proxy] configured for downloads: {PROXY_URL[:30]}...')
else:
    print('[proxy] no PROXY_URL configured \u2014 downloads may be rate-limited from Kaggle IPs')

with open(CONFIG_DIR / 'hf_repos.yaml') as f:
    repos_cfg = yaml.safe_load(f)

STAGE0_REPO = repos_cfg['repos']['stage0_codec']['repo_id']
HF_API      = HfApi(token=HF_TOKEN)
print(f'[config] stage0 repo: {STAGE0_REPO}')
print(f'[config] cookies: {"found at " + str(COOKIES_PATH) if COOKIES_PATH.exists() else "NOT FOUND \u2014 403s likely"}')

In [ ]:
def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        try:
            with open(CHECKPOINT_PATH) as f:
                state = json.load(f)
            print(f'[checkpoint] local \u2014 done={len(state["done"])} failed={len(state["failed"])} standardized={len(state["standardized"])}')
            return state
        except Exception:
            pass

    try:
        url = f'https://huggingface.co/datasets/{STAGE0_REPO}/resolve/main/checkpoint_p1b.json'
        r = requests.get(url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=30)
        if r.status_code == 200:
            state = r.json()
            with open(CHECKPOINT_PATH, 'w') as f:
                json.dump(state, f)
            print(f'[checkpoint] HF fallback \u2014 done={len(state["done"])}')
            return state
    except Exception:
        pass

    print('[checkpoint] fresh start')
    return {
        'done': [],
        'failed': [],
        'standardized': [],
        'stats': {
            'downloaded': 0,
            'standardized': 0,
            'failed_download': 0,
            'failed_standardize': 0,
            'too_short': 0,
            'too_small': 0,
            'cookie_rejected': 0,
        },
        'last_updated': None,
    }


cp_lock = threading.Lock()

def save_checkpoint(state, upload=False):
    with cp_lock:
        state['last_updated'] = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
        tmp = str(CHECKPOINT_PATH) + '.tmp'
        with open(tmp, 'w') as f:
            json.dump(state, f)
        os.replace(tmp, str(CHECKPOINT_PATH))

    if not upload:
        return

    for attempt in range(6):
        try:
            HF_API.upload_file(
                path_or_fileobj=json.dumps(state).encode(),
                path_in_repo='checkpoint_p1b.json',
                repo_id=STAGE0_REPO,
                repo_type='dataset',
                commit_message='p1b checkpoint',
            )
            return
        except Exception as e:
            wait = min(2 ** attempt, 60)
            print(f'[checkpoint] upload failed attempt {attempt+1}: {e} \u2014 retry in {wait}s')
            time.sleep(wait)


state = load_checkpoint()
done_set         = set(state['done'])
standardized_set = set(state['standardized'])
# Ensure cookie_rejected stat key exists in older checkpoints
if 'cookie_rejected' not in state['stats']:
    state['stats']['cookie_rejected'] = 0

In [ ]:
manifest_local = WORK_DIR / 'video_manifest.jsonl'

_batch_size = globals().get('BATCH_SIZE', 100)

if not manifest_local.exists():
    print('[manifest] downloading from HF...')
    for attempt in range(6):
        try:
            url = f'https://huggingface.co/datasets/{STAGE0_REPO}/resolve/main/video_manifest.jsonl'
            r = requests.get(url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=120, stream=True)
            r.raise_for_status()
            with open(manifest_local, 'wb') as f:
                for chunk in r.iter_content(chunk_size=65536):
                    f.write(chunk)
            print(f'[manifest] downloaded to {manifest_local}')
            break
        except Exception as e:
            wait = min(2 ** attempt, 60)
            print(f'[manifest] download attempt {attempt+1} failed: {e} \u2014 retry in {wait}s')
            time.sleep(wait)

all_videos = []
with open(manifest_local, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            all_videos.append(json.loads(line))

pending = [v for v in all_videos if v['video_id'] not in done_set]
pending = pending[:_batch_size]

print(f'[manifest] total={len(all_videos)} already_done={len(done_set)} pending={len(pending)} batch_size={_batch_size}')


In [ ]:
print_lock = threading.Lock()

def tprint(*args):
    with print_lock:
        print(*args, flush=True)


# --- Fix 2: Mid-session cookie refresh helper ---
def _refresh_cookies():
    """Re-copy cookies from the first valid COOKIE_CANDIDATES path to COOKIES_PATH.
    Called every COOKIE_REFRESH_EVERY downloads to pick up externally-updated cookies.
    """
    global COOKIES_SRC
    # Re-scan candidates in case a new one appeared
    for candidate in COOKIE_CANDIDATES:
        if candidate.exists():
            try:
                shutil.copy2(str(candidate), str(COOKIES_PATH))
                COOKIES_SRC = candidate
                tprint(f'[cookies] refreshed at download #{_download_counter} from {candidate}')
                return
            except Exception as e:
                tprint(f'[cookies] refresh failed from {candidate}: {e}')
    tprint('[cookies] WARNING: no cookie candidate found for mid-session refresh')


# --- Fix 4: Better error truncation (head + tail instead of tail-only) ---
def _truncate_error(err_full):
    """Truncate stderr to show both the beginning and end of the error message.
    The yt-dlp cookie error starts with context at the top and the identifying
    token (like 'w-do-i-pass-cookies-to-yt-dlp') appears early. Tail-only
    truncation was cutting off this identifying token.
    """
    if len(err_full) <= 300:
        return err_full
    return err_full[:150] + '...' + err_full[-150:]


def download_video(video):
    global _consecutive_cookie_rejections, _circuit_breaker_tripped

    vid_id   = video['video_id']
    url      = f'https://www.youtube.com/watch?v={vid_id}'
    out_path = DOWNLOAD_DIR / f'{vid_id}.wav'

    if out_path.exists() and out_path.stat().st_size > MIN_FILE_BYTES:
        return vid_id, 'already_exists', out_path

    # --- Circuit-breaker fast exit: if cookies are dead, don't even try ---
    with _cookie_lock:
        if _circuit_breaker_tripped:
            return vid_id, 'cookie_rejected', None

    cmd = [
        'yt-dlp',
        '--no-playlist',
        '--no-warnings',
        '--quiet',
        '--format', 'bestaudio/best/ba/w',
        '--extract-audio',
        '--audio-format', 'wav',
        '--audio-quality', '0',
        '--postprocessor-args', 'ffmpeg:-ar 24000 -ac 1',
        '--output', str(DOWNLOAD_DIR / f'{vid_id}.%(ext)s'),
        '--no-part',
        '--retries', '3',
        '--fragment-retries', '3',
        '--sleep-requests', '1',
    ]

    if COOKIES_PATH.exists():
        cmd.extend(['--cookies', str(COOKIES_PATH)])
    else:
        tprint(f'  [warning] No cookies file for {vid_id} \u2014 likely 403')

    if PROXY_URL:
        cmd.extend(['--proxy', PROXY_URL])

    cmd.append(url)

    for attempt in range(MAX_RETRIES):
        # --- Check circuit-breaker before each attempt ---
        with _cookie_lock:
            if _circuit_breaker_tripped:
                return vid_id, 'cookie_rejected', None

        try:
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)

            if result.returncode != 0:
                # Fix 4: head+tail truncation so cookie-rejection token is visible
                err_full = result.stderr.strip()
                err = _truncate_error(err_full)

                if 'Video unavailable' in err or 'Private video' in err or 'removed' in err.lower() or 'not available' in err.lower():
                    return vid_id, 'unavailable', None
                is_403 = '403' in err or 'Forbidden' in err
                is_format_err = 'Requested format is not available' in err
                is_geo_blocked = 'VPN or a proxy' in err or 'Zimbabwe' in err
                is_rate_limited = 'try again later' in err.lower() or 'rate limit' in err.lower()
                # --- Fix 1: Explicit cookie-rejection detection ---
                is_cookie_rejected = 'w-do-i-pass-cookies-to-yt-dlp' in err or ('cookies' in err.lower() and 'pass' in err.lower())

                if is_cookie_rejected:
                    with _cookie_lock:
                        _consecutive_cookie_rejections += 1
                        if _consecutive_cookie_rejections >= COOKIE_CIRCUIT_BREAKER:
                            _circuit_breaker_tripped = True
                        cookie_count = _consecutive_cookie_rejections

                    # On the very first rejection, try a cookie refresh + 120s cooldown
                    if cookie_count == 1:
                        tprint(f'[cookie-rejected] {vid_id} \u2014 YouTube rejected cookies (1st), refreshing cookies + 120s cooldown')
                        _refresh_cookies()
                        time.sleep(120)
                        continue
                    else:
                        # 2nd+ rejection \u2014 cookies are dead, skip immediately
                        tprint(f'[cookie-rejected] {vid_id} \u2014 cookies dead ({cookie_count} consecutive), skipping')
                        return vid_id, 'cookie_rejected', None

                if (is_403 or is_rate_limited) and attempt < MAX_RETRIES - 1:
                    wait = 15 * (attempt + 1)
                    tprint(f'  [403 retry] {vid_id} attempt {attempt+1}/{MAX_RETRIES} \u2014 waiting {wait}s')
                    time.sleep(wait)
                    continue
                if is_format_err:
                    return vid_id, 'format_unavailable', None
                if is_geo_blocked:
                    return vid_id, 'geo_blocked', None
                if is_rate_limited and attempt < MAX_RETRIES - 1:
                    wait = 30 * (attempt + 1)
                    tprint(f'  [rate-limited retry] {vid_id} attempt {attempt+1}/{MAX_RETRIES}')
                    time.sleep(wait)
                    continue
                if attempt == MAX_RETRIES - 1:
                    return vid_id, f'failed:{err}', None
                time.sleep(5 * (attempt + 1))
                continue

            if not out_path.exists():
                wav_candidates = list(DOWNLOAD_DIR.glob(f'{vid_id}.*'))
                if wav_candidates:
                    wav_candidates[0].rename(out_path)
                else:
                    return vid_id, 'no_output_file', None

            if out_path.stat().st_size < MIN_FILE_BYTES:
                out_path.unlink(missing_ok=True)
                return vid_id, 'too_small', None

            # Successful download resets the consecutive counter
            with _cookie_lock:
                _consecutive_cookie_rejections = 0

            return vid_id, 'ok', out_path

        except subprocess.TimeoutExpired:
            if attempt == MAX_RETRIES - 1:
                return vid_id, 'timeout', None
            time.sleep(10)
        except Exception as e:
            if attempt == MAX_RETRIES - 1:
                return vid_id, f'exception:{e}', None
            time.sleep(5 * (attempt + 1))

    return vid_id, 'max_retries', None


def standardize_audio(vid_id, input_path):
    out_path = STANDARD_DIR / f'{vid_id}.wav'

    if out_path.exists() and out_path.stat().st_size > MIN_FILE_BYTES:
        return vid_id, 'already_exists', out_path

    try:
        with sf.SoundFile(str(input_path)) as f:
            duration = len(f) / f.samplerate

        if duration < MIN_DUR_SEC:
            return vid_id, 'too_short', None

        result = subprocess.run(
            [
                'ffmpeg', '-y',
                '-i', str(input_path),
                '-ar', str(TARGET_SR),
                '-ac', '1',
                '-sample_fmt', 's16',
                '-vn',
                str(out_path),
            ],
            capture_output=True,
            timeout=600,
        )

        if result.returncode != 0:
            # Fix 4: head+tail truncation for ffmpeg errors too
            err_full = result.stderr.decode().strip()
            err = _truncate_error(err_full)
            return vid_id, f'ffmpeg_error:{err}', None

        if out_path.stat().st_size < MIN_FILE_BYTES:
            out_path.unlink(missing_ok=True)
            return vid_id, 'output_too_small', None

        input_path.unlink(missing_ok=True)
        return vid_id, 'ok', out_path

    except Exception as e:
        return vid_id, f'standardize_exception:{e}', None


def download_and_standardize(video):
    global _download_counter

    vid_id, dl_status, dl_path = download_video(video)

    time.sleep(RATE_LIMIT_DELAY)

    # --- Fix 2: Mid-session cookie refresh ---
    with _cookie_lock:
        _download_counter += 1
        if _download_counter % COOKIE_REFRESH_EVERY == 0:
            _refresh_cookies()

    if dl_status not in ('ok', 'already_exists'):
        return vid_id, dl_status, None, 'download'

    if vid_id in standardized_set:
        std_path = STANDARD_DIR / f'{vid_id}.wav'
        if std_path.exists():
            return vid_id, 'already_standardized', std_path, 'standardize'

    vid_id, std_status, std_path = standardize_audio(vid_id, dl_path)
    if std_status == 'ok' and dl_path and dl_path.exists():
        try:
            dl_path.unlink(missing_ok=True)
        except Exception:
            pass
    return vid_id, std_status, std_path, 'standardize'

In [ ]:
total     = len(pending)
completed = 0
success   = 0

print(f'[download] starting {total} videos with {DOWNLOAD_WORKERS} workers')
print(f'[download] cookie refresh every {COOKIE_REFRESH_EVERY} downloads, circuit-breaker at {COOKIE_CIRCUIT_BREAKER} consecutive cookie rejections\n')

with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
    futures = {executor.submit(download_and_standardize, v): v for v in pending}

    for future in as_completed(futures):
        vid_id, status, out_path, stage = future.result()
        completed += 1

        with cp_lock:
            if status in ('ok', 'already_exists', 'already_standardized'):
                success += 1
                if vid_id not in done_set:
                    done_set.add(vid_id)
                    state['done'].append(vid_id)
                if stage == 'standardize' and vid_id not in standardized_set:
                    standardized_set.add(vid_id)
                    state['standardized'].append(vid_id)
                    state['stats']['standardized'] += 1
                state['stats']['downloaded'] += 1
            else:
                # Fix 1: Track cookie_rejected in stats
                if status == 'cookie_rejected':
                    state['stats']['cookie_rejected'] += 1
                else:
                    if vid_id not in state['failed']:
                        state['failed'].append(vid_id)
                    if stage == 'download':
                        state['stats']['failed_download'] += 1
                        if status == 'too_small':
                            state['stats']['too_small'] += 1
                    elif stage == 'standardize':
                        state['stats']['failed_standardize'] += 1
                        if status == 'too_short':
                            state['stats']['too_short'] += 1


        if _circuit_breaker_tripped:
            tprint(f'\n[circuit-breaker] {COOKIE_CIRCUIT_BREAKER}+ consecutive cookie rejections \u2014 cookies are fully expired. Stopping p1b early.')
            # Cancel remaining futures
            for f in futures:
                f.cancel()
            save_checkpoint(state, upload=True)
            raise CookieExpiredError(
                f'{COOKIE_CIRCUIT_BREAKER}+ consecutive cookie-rejected downloads \u2014 cookies expired'
            )

        if completed % SAVE_EVERY == 0 or completed == total:
            upload_now = completed % (SAVE_EVERY * 10) == 0
            save_checkpoint(state, upload=upload_now)
            tprint(f'  [{completed}/{total}] ok={success} failed={len(state["failed"])} | '
                   f'dl={state["stats"]["downloaded"]} std={state["stats"]["standardized"]} '
                   f'cookie_rej={state["stats"]["cookie_rejected"]}')

        # --- Fix 5: Distinct log line for cookie_rejected ---
        if status == 'cookie_rejected':
            tprint(f'  [cookie-expired] {vid_id} \u2014 cookies rejected by YouTube')
        elif status not in ('ok', 'already_exists', 'already_standardized'):
            tprint(f'  [skip] {vid_id} \u2014 {status}')

In [ ]:
print('\n[summary]')
print(f'  total processed  : {completed}')
print(f'  standardized ok  : {state["stats"]["standardized"]}')
print(f'  failed download  : {state["stats"]["failed_download"]}')
print(f'  failed standardize: {state["stats"]["failed_standardize"]}')
print(f'  too short        : {state["stats"]["too_short"]}')
print(f'  too small        : {state["stats"]["too_small"]}')
print(f'  cookie rejected  : {state["stats"]["cookie_rejected"]}')

if state['failed']:
    with open(FAILED_LOG, 'w') as f:
        for vid_id in state['failed']:
            f.write(vid_id + '\n')
    print(f'  failed log       : {FAILED_LOG} ({len(state["failed"])} entries)')

ready_files = list(STANDARD_DIR.glob('*.wav'))
total_gb = sum(f.stat().st_size for f in ready_files) / 1024**3
print(f'\n[output] {len(ready_files)} WAV files ready in {STANDARD_DIR}')
print(f'[output] total size: {total_gb:.2f} GB')

save_checkpoint(state, upload=True)
print('\n[done] checkpoint saved \u2014 ready for p1c_clean_cpu.ipynb')